## Field project setup

This notebook demonstrates options for creating a field project.  It shows how to create a folder and group for project items.  Then shows creating a track view and map for viewing the current location of team members.  It then demonstrates three approaches for map and layer creation: in code, from a Survey, and with clone_items from a template map item.  Finally, it creates offline areas for the map for project areas defined in a layers.

### Connect to the GIS and get started

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection
from arcgis.mapping import WebMap
import pandas as pd

gis = GIS('home')

PROJECT_NAME = 'New project for UC demo'
PROJECT_FOLDER = 'UC 2026 demo'
PROJECT_MEMBERS = ['jakebrown_Nitro', 'Colin_Nitro']

### Setup users

In [ ]:
USERS_CSV = "user_list.csv"
df = pd.read_csv(USERS_CSV)

users = []

for index, row in df.iterrows():
    u = gis.users.get(row["Username"])
    if u is None:
        users.append(gis.users.create(
            username=row["Username"], 
            password=row["Password"],
            firstname=row["First Name"],
            lastname=row["Last Name"],
            email=row["Email"],
            role=row["Role"],
            user_type=row["User Type"]
        ))
    else:
        users.append(u)

### Create folder and group

In [ ]:
# Reuse folder if it already exists, otherwise create it
project_folder = gis.content.folders.get(folder=PROJECT_FOLDER)
if project_folder is None:
    project_folder = gis.content.folders.create(folder=PROJECT_FOLDER)

# Reuse group if it already exists, otherwise create it
group_title = PROJECT_NAME + " Group"
existing_groups = gis.groups.search(
    query=f'title:"{group_title}" AND owner:{gis.users.me.username}',
    max_groups=10
)
project_group = next((g for g in existing_groups if g.title == group_title), None)

if project_group is None:
    project_group = gis.groups.create(
        title=group_title,
        tags="demo",
        description="Group to share project info",
        access="org",
        is_invitation_only=False,
    )

# Add only users not already in the group
current_usernames = project_group.get_members().get("users", [])
to_add = [u for u in PROJECT_MEMBERS if u not in current_usernames]
if to_add:
    project_group.add_users(to_add)

project_group

### Create a track view and a map

In [ ]:
track_view_name = PROJECT_NAME + ' track view'
track_view = gis.admin.location_tracking.create_track_view(track_view_name)

track_view.mobile_users.add(PROJECT_MEMBERS)
track_view.mobile_users.list()

In [ ]:
# Users need to have a role that includes "portal:user:joinGroup" and "portal:user:viewTracks" to view tracks

track_view.viewers.add(PROJECT_MEMBERS)
track_view.viewers.list()

In [ ]:
# Create a new map, add the last known location layer and share with the group

lkl_map = gis.map()
lkl_map.content.add(track_view.last_known_locations_layer)

lkl_webmap_item_properties = {'title':PROJECT_NAME + " team locations map",
                              'snippet':'Map created using Python API for ' + PROJECT_NAME,
                              'tags':['demo']
                             }
map_item = lkl_map.save(lkl_webmap_item_properties, folder=PROJECT_FOLDER)

# share the map with the group
map_item.sharing.groups.add(group=project_group)

map_item

### Map and layer creation options

#### In code

In [ ]:
# service parameters
create_params = {
    "maxRecordCount": 2000,
    "supportedQueryFormats": "JSON",
    "capabilities": "Query,Create,Update,Delete,Editing",
    "description": "Service created from the Python API",
    "allowGeometryUpdates": True,
    "hasStaticData": True,
    "units": "esriMeters",
    "syncEnabled": False,
    "editorTrackingInfo": {
        "enableEditorTracking": False,
        "enableOwnershipAccessControl": False,
        "allowOthersToQuery": True,
        "allowOthersToUpdate": True,
        "allowOthersToDelete": False,
        "allowAnonymousToUpdate": True,
        "allowAnonymousToDelete": False,
    },
    "xssPreventionInfo": {
        "xssPreventionEnabled": True,
        "xssPreventionRule": "InputOnly",
        "xssInputRule": "rejectInvalid",
    },
    "initialExtent": {
        "spatialReference": {"wkid": 4326},
        "xmin": -118.84764718980026,
        "ymin": 33.99799168307417,
        "xmax": -118.7618165013238,
        "ymax": 34.026450333167524,
    },
    "spatialReference": {"wkid": 4326},
    "tables": [],
    "name": "Inspection Project Layers",
}

# define layer props
layer_schema = {
    "layers": [
        {
            "name": "Inspection Assets",
            "type": "Feature Layer",
            "defaultVisibility": True,
            "relationships": [],
            "isDataVersioned": False,
            "supportsRollbackOnFailureParameter": True,
            "supportsAdvancedQueries": False,
            "geometryType": "esriGeometryPoint",
            "minScale": 0,
            "maxScale": 0,
            "extent": {
                "xmin": -134.74729261792592,
                "ymin": 23.56096242376989,
                "xmax": -55.695547615409396,
                "ymax": 50.309217030288835,
                "spatialReference": {"wkid": 4326},
            },
            "drawingInfo": {
                "transparency": 0,
                "labelingInfo": None,
                "renderer": {
                    "type": "simple",
                    "symbol": {
                        "color": [20, 158, 206, 130],
                        "size": 18,
                        "angle": 0,
                        "xoffset": 0,
                        "yoffset": 0,
                        "type": "esriSMS",
                        "style": "esriSMSCircle",
                        "outline": {
                            "color": [255, 255, 255, 220],
                            "width": 2.25,
                            "type": "esriSLS",
                            "style": "esriSLSSolid",
                        },
                    },
                },
            },
            "allowGeometryUpdates": True,
            "hasAttachments": True,
            "htmlPopupType": "esriServerHTMLPopupTypeNone",
            "hasM": False,
            "hasZ": False,
            "objectIdField": "OBJECTID",
            "formInfo": {
                "formElements": [
                ]
            },
            "fields": [
                {
                    "name": "OBJECTID",
                    "type": "esriFieldTypeOID",
                    "alias": "OBJECTID",
                    "sqlType": "sqlTypeOther",
                    "nullable": False,
                    "editable": False,
                    "domain": None,
                    "defaultValue": None,
                },
                {
                    "name": "id",
                    "type": "esriFieldTypeInteger",
                    "alias": "id",
                    "sqlType": "sqlTypeInteger",
                    "nullable": True,
                    "editable": True,
                    "domain": None,
                    "defaultValue": None,
                },
                {
                    "name": "name",
                    "type": "esriFieldTypeString",
                    "alias": "name",
                    "sqlType": "sqlTypeNVarchar",
                    "nullable": True,
                    "editable": True,
                    "domain": None,
                    "defaultValue": None,
                    "length": 256,
                },
                {
                    "name": "rating",
                    "type": "esriFieldTypeString",
                    "alias": "rating",
                    "sqlType": "sqlTypeNVarchar",
                    "nullable": True,
                    "editable": True,
                    "domain": None,
                    "defaultValue": None,
                    "length": 256,
                },
            ],
            "templates": [
                {
                    "name": "New Feature",
                    "description": "",
                    "drawingTool": "esriFeatureEditToolPoint",
                    "prototype": {
                        "attributes": {
                            "id": None,
                            "name": None,
                            "rating": None,
                        }
                    },
                }
            ],
            "supportedQueryFormats": "JSON",
            "hasStaticData": True,
            "maxRecordCount": 10000,
            "capabilities": "Query,Create,Update,Delete,Editing",
        }
    ]
}


# create the service
new_service = gis.content.create_service(
    name="Inspection Project Layers",
    create_params=create_params,
    tags="demo",
)

# Add layer definition and schema
new_feature_layer = FeatureLayerCollection.fromitem(new_service)
new_feature_layer.manager.add_to_definition(layer_schema)


inspection_map = gis.map()
inspection_map.content.add(new_feature_layer.layers[0])

inspection_webmap_item_properties = {"title":PROJECT_NAME + " Inspection map",
                              "snippet":"Map created using Python API for " + PROJECT_NAME,
                              "tags":["demo"]
                             }

inspection_map_item = inspection_map.save(inspection_webmap_item_properties, folder=PROJECT_FOLDER)


#### From a Survey

In [ ]:
from arcgis.apps.survey123 import SurveyManager
survey_manager = SurveyManager(gis)

new_survey = survey_manager.create(
    title = "Inspection Project Survey",
    folder = PROJECT_FOLDER,
    tags = "Survey123, Python",
    description = "This survey is...", 
    thumbnail = r"thumbnail.png"
)

published_survey = new_survey.publish(
    xlsform=r"Inspection Project Template Survey.xlsx", 
    info=
        {"queryInfo": {
            "mode": "manual",
            "editEnabled": True,
            "copyEnabled": False
            }
        }, 
    enable_delete_protection=True
    )

#### With clone_items

In [ ]:
template_map = gis.content.get('f345a96e75fa4444acd8d8250a3f290b')
template_map

In [ ]:
cloned_items = gis.content.clone_items(items=[template_map],
                                       folder=PROJECT_FOLDER)
cloned_items

In [ ]:
map_item = next((item for item in cloned_items if item.type == "Web Map"), None)

if map_item is not None:    
    webmap_item_properties = {'title':PROJECT_NAME + ' Inspection Map',
                            'snippet':'Map created using Python API for ' + PROJECT_NAME,
                            'tags':['demo']
                            }
    map_item.update(webmap_item_properties)

    map_item.sharing.groups.add(group=project_group)

    map_item

### Create offline areas

In [ ]:
new_offline_map = WebMap(item=map_item)

# The item id of the feature layer to use for the areas you want to create
# If the feature layer has more than 16 features, only the first 16 features will be queried

# Portland Administrative Sextants
FEATURE_LAYER_ITEM_ID = '12e3f9dadda048e993d504362cf815b4'
# The id of the layer to use
FEATURE_LAYER_ID = 0
# Field name of the attribute to use to name the areas that are created
AREA_NAME_ATTRIBUTE = 'Sextant'

# Properties for the output areas
output_title_template = '{} Inspection Area'
output_snippet_template = 'A map that contains project data for {} inspection area.'
output_tags = 'demo'

offline_areas_item = gis.content.get(FEATURE_LAYER_ITEM_ID)
offline_areas = offline_areas_item.layers[FEATURE_LAYER_ID].query(result_record_count=16, return_all_records=False)
    
for offline_area in offline_areas.features:
    area_name = offline_area.attributes[AREA_NAME_ATTRIBUTE]

    print('Creating offline map area for ' + area_name)
    
    item_prop = {'title': output_title_template.format(area_name),
                 'snippet': output_snippet_template.format(area_name),
                 'tags': [output_tags]}

    try:
        map_area = new_offline_map.offline_areas.create(
            area=offline_area.geometry,
            item_properties=item_prop,
            folder=PROJECT_FOLDER,
        )
    except Exception as ex:
        print(f"Failed creating map area for {area_name}: {ex}")
